# Dựng dữ liệu public (MSnLib / MassBank / NPLIB1) cho MARASON trên Kaggle

Notebook chạy `data_scripts/public/` của nhánh `claude/magical-knuth-oswt8t` (repo `Giamyy/ms-pred`):

1. `download_public_datasets.sh`: tải dữ liệu gốc (MSnLib từ Zenodo, MassBank từ GitHub release, NPLIB1 từ Zenodo).
2. `create_public_dataset.py`: chuyển thành `labels.tsv`, `spec_files.hdf5`, `splits/`, `audit.json`
   (và `retrieval/` cho nplib1).
3. `data_scripts/dag/run_magma.sh`: MAGMa (`magma_outputs/`) và subformula (`subformulae/no_subform.hdf5`).

Bước 1–2 nhanh. **Bước 3 (MAGMa) là bước chậm, chỉ dùng CPU**, nên notebook này chạy với *Accelerator: None*
(không tốn quota GPU) và chia MAGMa thành nhiều shard khi một phiên 12 giờ không đủ.
MAGMa xử lý từng mục độc lập, nên chạy từng phần rồi gộp cho cùng kết quả như chạy một lần.

| `STAGE` | Làm gì |
|---|---|
| `"test"` | Dựng dataset, chạy MAGMa cho `TEST_N` mục đầu để **đo thời gian** và ước lượng số shard cần |
| `"all"` | Làm hết trong một phiên (đủ cho **nplib1**; massbank/msnlib xem kết quả `"test"` trước) |
| `"shard"` | Dựng dataset, chạy MAGMa cho phần `SHARD_ID` trong `NUM_SHARDS` phần |
| `"merge"` | Dựng dataset, gộp MAGMa của các shard (*Add Input* output các notebook shard), xuất dataset đầy đủ |

**Cách chạy:** *Accelerator: None*, *Internet: On*.
1. `STAGE = "test"` → *Run All* (chạy tương tác), xem dòng `Ước lượng`.
2. Nếu ước lượng dưới khoảng 10 giờ: `STAGE = "all"` → *Save Version → Save & Run All*.
   Nếu không: `NUM_SHARDS` bản sao với `STAGE = "shard"`, `SHARD_ID = 0 … NUM_SHARDS-1` (chạy song song),
   rồi một notebook `STAGE = "merge"` với *Add Input* output của mọi notebook shard.
3. Output (`ms-pred/data/spec_datasets/<DATASET>/`) dùng thẳng cho notebook train
   `notebooks/kaggle/marason_public_kaggle.ipynb` (nhánh `claude/magical-knuth-oswt8t`): *Add Input* output này rồi đặt
   `RESUME_FROM = "/kaggle/input/<tên notebook này>/ms-pred"`. Notebook train sẽ thấy dữ liệu đã sẵn sàng và bỏ qua bước dựng.

In [ ]:
# ===== CẤU HÌNH =====
DATASET      = "nplib1"           # "nplib1" | "massbank" | "msnlib"
STAGE        = "test"             # "test" | "all" | "shard" | "merge"
NUM_SHARDS   = 4                  # giống nhau ở mọi notebook shard và merge
SHARD_ID     = 0                  # STAGE = "shard": 0 .. NUM_SHARDS-1
SHARD_INPUTS = ["/kaggle/input"]  # STAGE = "merge": nơi tìm output các notebook shard (tìm đệ quy)
TEST_N       = 300                # STAGE = "test": số mục chạy thử MAGMa

REPO_URL     = "https://github.com/Giamyy/ms-pred.git"
REPO_BRANCH  = "claude/magical-knuth-oswt8t"   # nhánh có data_scripts/public
# Phiên bản dữ liệu gốc: None = mặc định trong download_public_datasets.sh. Mọi notebook shard/merge phải dùng cùng giá trị.
MSNLIB_RECORD    = None           # vd "21105617"
MASSBANK_RELEASE = None           # vd "2026.11"

In [ ]:
import os, subprocess, sys, json, shutil, time, hashlib
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = Path("/tmp/ms-pred")       # repo, .venv và dữ liệu gốc nằm ngoài /kaggle/working (không bị lưu vào output)
NCPU = os.cpu_count() or 4
assert DATASET in {"nplib1", "massbank", "msnlib"}, DATASET
assert STAGE in {"test", "all", "shard", "merge"}, STAGE
assert 0 <= SHARD_ID < NUM_SHARDS

os.environ["raw_dir"] = "/tmp/raw"
os.environ["workers"] = str(NCPU)
if MSNLIB_RECORD:
    os.environ["MSNLIB_RECORD"] = str(MSNLIB_RECORD)
if MASSBANK_RELEASE:
    os.environ["MASSBANK_RELEASE"] = str(MASSBANK_RELEASE)

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh shell, in output ngay khi có; bấm Interrupt sẽ dừng cả tiến trình con."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", start_new_session=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    p.tail = []                                       # 200 dòng cuối, để tóm tắt lỗi
    try:
        for line in p.stdout:
            p.tail = (p.tail + [line])[-200:]
            if "it/s]" in line or "s/it]" in line:      # bỏ bớt dòng tiến độ tqdm
                continue
            print(line, end="")
    except KeyboardInterrupt:
        os.killpg(p.pid, 9)
        raise
    if p.wait() != 0 and check:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): {cmd}")
    return p

import urllib.request
try:
    urllib.request.urlopen("https://zenodo.org", timeout=15)
    print("Internet OK")
except Exception as e:
    raise SystemExit(f"Chưa có internet, hãy bật Settings -> Internet -> On.\n{e}")
print("CPU cores:", NCPU, "| DATASET:", DATASET, "| STAGE:", STAGE)

In [ ]:
# ===== 1. Clone repo + cài môi trường CPU (~10 phút) =====
if not REPO.exists():
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO}")
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
PY = str(REPO / ".venv" / "bin" / "python")
CHECK = "import sys, rdkit, h5py, ms_pred; from ms_pred.magma import fragmentation"
def env_ready():
    # .venv có thể còn sót lại (rỗng hoặc thiếu gói) từ một lần uv sync lỗi trước: kiểm tra bằng import thật
    return Path(PY).exists() and subprocess.run([PY, "-c", CHECK], capture_output=True).returncode == 0
if not env_ready():
    sh("pip install -q -U uv && uv --version")
    os.environ.setdefault("UV_HTTP_TIMEOUT", "300")    # mạng Kaggle đôi khi chậm khi tải torch / dgl
    # --python 3.11: Python 3.13 của Kaggle không có wheel torch-scatter / torch-sparse dựng sẵn
    for attempt in range(3):
        p = sh("uv sync --extra cpu --python 3.11", cwd=REPO, check=False)
        if p.returncode == 0:
            break
        errs = [l.rstrip() for l in p.tail if any(w in l.lower() for w in ("error", "failed", "caused by", "hint", "×", "╰─"))]
        print(f"\n===== uv sync lỗi (lần {attempt + 1}/3). Các dòng lỗi: =====")
        print("\n".join(errs[-25:]) or "".join(p.tail[-25:]))
        pyproject = REPO / "pyproject.toml"
        text = pyproject.read_text()
        if attempt == 0 and "pygmtools = { git" in text:
            # pygmtools được cài từ GitHub (git clone), hay lỗi trên Kaggle; bản PyPI đủ dùng cho bước dựng dữ liệu
            pyproject.write_text("\n".join(l for l in text.splitlines() if not l.startswith("pygmtools = { git")) + "\n")
            print("Thử lại với pygmtools từ PyPI thay vì GitHub")
        time.sleep(10)
    else:
        raise RuntimeError("uv sync thất bại: xem các dòng lỗi ở trên (gửi ảnh chụp phần đó để sửa)")
sh(f"{PY} -c \"{CHECK}; print('python', sys.version.split()[0], '| rdkit', rdkit.__version__)\"")
# Các script của tác giả gọi `python`/`python3`: cho chúng dùng môi trường vừa cài
os.environ["PATH"] = f"{REPO / '.venv' / 'bin'}:{os.environ['PATH']}"

Hai ô dưới ghi vào repo hai script tách và gộp shard MAGMa (nhánh `claude/magical-knuth-oswt8t` chưa có).
Shard `i` lấy các dòng `i, i+k, i+2k…` của `labels.tsv` cùng phổ tương ứng trong `spec_files.hdf5`.

In [ ]:
%%writefile /tmp/ms-pred/data_scripts/dag/make_magma_shard.py
#!/usr/bin/env python3
"""Cut one shard out of a dataset so MAGMA can run on it in a separate job.

Shard ``i`` of ``k`` takes rows ``i, i+k, i+2k, ...`` of the source labels.tsv
and copies the matching ``{spec}.ms`` entries of spec_files.hdf5. Run
``bash data_scripts/dag/run_magma.sh <output dataset name>`` on the result,
then combine the shards with data_scripts/dag/merge_magma_shards.py.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import h5py
import pandas as pd


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source-dataset", type=Path, required=True,
                        help="Dataset dir with labels.tsv and spec_files.hdf5")
    parser.add_argument("--output-dataset", type=Path, required=True)
    parser.add_argument("--num-shards", type=int, required=True)
    parser.add_argument("--shard-id", type=int, required=True)
    return parser.parse_args()


def main():
    args = get_args()
    if not 0 <= args.shard_id < args.num_shards:
        raise ValueError(f"--shard-id must be in [0, {args.num_shards})")

    labels = pd.read_csv(args.source_dataset / "labels.tsv", sep="\t")
    shard = labels.iloc[args.shard_id::args.num_shards]

    args.output_dataset.mkdir(parents=True, exist_ok=True)
    shard.to_csv(args.output_dataset / "labels.tsv", sep="\t", index=False)

    missing = []
    with h5py.File(args.source_dataset / "spec_files.hdf5", "r") as src, \
            h5py.File(args.output_dataset / "spec_files.hdf5", "w") as dst:
        for spec in shard["spec"].astype(str):
            key = f"{spec}.ms"
            if key in src:
                src.copy(key, dst, name=key)
            else:
                missing.append(spec)

    summary = {
        "source_specs": int(len(labels)),
        "shard_id": args.shard_id,
        "num_shards": args.num_shards,
        "shard_specs": int(len(shard)),
        "missing_in_spec_files": len(missing),
    }
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /tmp/ms-pred/data_scripts/dag/merge_magma_shards.py
#!/usr/bin/env python3
"""Merge MAGMA and subformula outputs computed on dataset shards.

Each shard dir (see make_magma_shard.py) holds
``magma_outputs/magma_tree.hdf5`` and ``subformulae/no_subform.hdf5``. Shards
cover disjoint spectra, so merging copies every top-level entry into one file:

* magma_tree.hdf5 is a PredSpecDB: root attributes are copied from the first
  shard (all shards must agree) and the embedded manifest is rebuilt from the
  merged groups.
* no_subform.hdf5 holds one ``{spec}_collision {ce}.json`` string per entry.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import h5py

from ms_pred.common.misc_utils import (
    _MANIFEST_GROUP,
    _build_manifest_from_old_structure,
    _embed_manifest_into_h5,
)

MAGMA_REL = Path("magma_outputs/magma_tree.hdf5")
SUBFORM_REL = Path("subformulae/no_subform.hdf5")


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--shard-dirs", type=Path, nargs="+", required=True)
    parser.add_argument("--output-dir", type=Path, required=True,
                        help="Dataset dir that receives magma_outputs/ and subformulae/")
    parser.add_argument("--overwrite", action="store_true")
    return parser.parse_args()


def copy_entries(shard_files: list[Path], out_path: Path, skip: set[str]) -> dict:
    written, duplicates = 0, []
    with h5py.File(out_path, "a") as dst:
        for path in shard_files:
            with h5py.File(path, "r") as src:
                for key in src.keys():
                    if key in skip:
                        continue
                    if key in dst:
                        duplicates.append(key)
                        continue
                    src.copy(key, dst, name=key)
                    written += 1
    if duplicates:
        raise ValueError(f"{len(duplicates)} entries appear in more than one shard, e.g. {duplicates[:5]}")
    return {"entries": written}


def merge_magma(shard_files: list[Path], out_path: Path) -> dict:
    with h5py.File(shard_files[0], "r") as first:
        root_attrs = dict(first.attrs)
    for path in shard_files[1:]:
        with h5py.File(path, "r") as f:
            if dict(f.attrs) != root_attrs:
                raise ValueError(f"Root attributes of {path} differ from {shard_files[0]}")

    with h5py.File(out_path, "w") as dst:
        dst.attrs.update(root_attrs)
    summary = copy_entries(shard_files, out_path, skip={_MANIFEST_GROUP})

    with h5py.File(out_path, "a") as dst:
        manifest = _build_manifest_from_old_structure(dst)
        _embed_manifest_into_h5(dst, manifest, out_path)
    summary["manifest_entries"] = len(manifest["leaf_path"])
    return summary


def merge_subformulae(shard_files: list[Path], out_path: Path) -> dict:
    with h5py.File(out_path, "w"):
        pass
    return copy_entries(shard_files, out_path, skip=set())


def main():
    args = get_args()

    outputs = {MAGMA_REL: args.output_dir / MAGMA_REL, SUBFORM_REL: args.output_dir / SUBFORM_REL}
    for rel in outputs:
        missing = [str(d) for d in args.shard_dirs if not (d / rel).exists()]
        if missing:
            raise FileNotFoundError(f"{rel} missing in shard dirs: {missing}")
    for out in outputs.values():
        if out.exists() and not args.overwrite:
            raise FileExistsError(f"{out} exists; pass --overwrite to replace it")
        out.parent.mkdir(parents=True, exist_ok=True)

    summary = {
        "shards": [str(d) for d in args.shard_dirs],
        "magma_tree": merge_magma([d / MAGMA_REL for d in args.shard_dirs], outputs[MAGMA_REL]),
        "no_subform": merge_subformulae([d / SUBFORM_REL for d in args.shard_dirs], outputs[SUBFORM_REL]),
    }
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
# ===== 2. Tải + chuyển dữ liệu (skip_magma=1: MAGMa chạy ở bước 3) =====
DS_REL = f"data/spec_datasets/{DATASET}"
DS_DIR = REPO / DS_REL
t0 = time.time()
sh(f"skip_magma=1 bash data_scripts/public/build_public_datasets.sh {DATASET}", cwd=REPO)
audit = json.loads((DS_DIR / "audit.json").read_text())
for k in ("entries", "spectra", "unique_inchikey_2d", "spectra_without_collision_energy", "adducts", "instruments", "splits", "dropped"):
    print(f"{k:34s}", audit.get(k))
LABELS_SHA = hashlib.sha256((DS_DIR / "labels.tsv").read_bytes()).hexdigest()
print(f"labels.tsv sha256 {LABELS_SHA[:16]}… | bước 2: {(time.time() - t0) / 60:.1f} phút")

In [ ]:
# ===== 3. MAGMa + subformula (CPU, bước lâu nhất) =====
def run_magma(dataset_name):
    # data_scripts/dag/run_magma.sh: MAGMa (--max-peaks 50 --ppm-diff 20) rồi 01_assign_subformulae.py --use-all
    sh(f"workers={NCPU} bash data_scripts/dag/run_magma.sh {dataset_name}", cwd=REPO)

def cut(name, num_shards, shard_id):
    """Cắt shard_id / num_shards của dataset vào data/spec_datasets/<name> (labels.tsv + đúng các phổ đó)."""
    out = REPO / "data/spec_datasets" / name
    shutil.rmtree(out, ignore_errors=True)
    sh(f"{PY} data_scripts/dag/make_magma_shard.py --source-dataset {DS_REL} --output-dataset data/spec_datasets/{name} "
       f"--num-shards {num_shards} --shard-id {shard_id}", cwd=REPO)
    return out

n_total = len(pd.read_csv(DS_DIR / "labels.tsv", sep="\t"))
t0 = time.time()

if STAGE == "all":
    # create_public_dataset.py giữ magma_outputs / subformulae cũ; xoá để chạy lại từ đầu với dữ liệu vừa dựng
    for old in ("magma_outputs", "subformulae"):
        shutil.rmtree(DS_DIR / old, ignore_errors=True)
    run_magma(DATASET)

elif STAGE == "test":
    # Chạy thử trên khoảng TEST_N mục rải đều trong dataset (một shard nhỏ)
    test_dir = cut(f"{DATASET}_test", max(n_total // TEST_N, 1), 0)
    run_magma(test_dir.name)
    minutes = (time.time() - t0) / 60
    n_test = len(pd.read_csv(test_dir / "labels.tsv", sep="\t"))
    total_h = minutes / n_test * n_total / 60
    print(f"\nƯớc lượng: {minutes:.1f} phút cho {n_test} mục -> khoảng {total_h:.1f} giờ cho {n_total} mục với {NCPU} CPU")
    print(f"Chia {NUM_SHARDS} shard: khoảng {total_h / NUM_SHARDS:.1f} giờ mỗi shard (một phiên Kaggle tối đa 12 giờ)")
    print("Gợi ý:", "STAGE = 'all' là đủ" if total_h < 10 else f"NUM_SHARDS >= {int(total_h / 10) + 1}")

elif STAGE == "shard":
    shard_dir = cut(f"{DATASET}_shard{SHARD_ID}", NUM_SHARDS, SHARD_ID)
    run_magma(shard_dir.name)
    (shard_dir / "spec_files.hdf5").unlink()        # phiên merge tự dựng lại, không cần lưu
    (shard_dir / "shard.json").write_text(json.dumps({
        "dataset": DATASET, "shard_id": SHARD_ID, "num_shards": NUM_SHARDS, "labels_sha256": LABELS_SHA,
        "entries": int(len(pd.read_csv(shard_dir / "labels.tsv", sep="\t"))), "commit": COMMIT}))

elif STAGE == "merge":
    found = sorted({p.parent for root in SHARD_INPUTS for p in Path(root).glob(f"**/{DATASET}_shard*/shard.json")})
    metas = [json.loads((d / "shard.json").read_text()) for d in found]
    for d, m in zip(found, metas):
        print("Tìm thấy:", d, {k: m[k] for k in ("shard_id", "num_shards", "entries")})
    if sorted(m["shard_id"] for m in metas) != list(range(NUM_SHARDS)) or any(m["num_shards"] != NUM_SHARDS for m in metas):
        raise RuntimeError(f"Cần đủ {NUM_SHARDS} shard (0..{NUM_SHARDS - 1}) của {DATASET}, tìm thấy {[m['shard_id'] for m in metas]}")
    if any(m["labels_sha256"] != LABELS_SHA for m in metas):
        raise RuntimeError("labels.tsv của shard khác labels.tsv vừa dựng: dữ liệu gốc đã đổi phiên bản? "
                           "Đặt MSNLIB_RECORD / MASSBANK_RELEASE giống lúc chạy shard")
    covered = set().union(*(set(pd.read_csv(d / "labels.tsv", sep="\t")["spec"].astype(str)) for d in found))
    expected = set(pd.read_csv(DS_DIR / "labels.tsv", sep="\t")["spec"].astype(str))
    if covered != expected:
        raise RuntimeError(f"Các shard phủ {len(covered)} mục, labels.tsv có {len(expected)}")
    for old in ("magma_outputs", "subformulae"):
        shutil.rmtree(DS_DIR / old, ignore_errors=True)
    sh(f"{PY} data_scripts/dag/merge_magma_shards.py --shard-dirs {' '.join(map(str, found))} "
       f"--output-dir {DS_REL} --overwrite", cwd=REPO)
print(f"Bước 3: {(time.time() - t0) / 60:.1f} phút")

In [ ]:
# ===== 4. Kiểm tra + xuất ra /kaggle/working =====
import h5py
if STAGE == "test":
    print("STAGE = test: không xuất dữ liệu. Đổi STAGE theo gợi ý ở trên.")
elif STAGE == "shard":
    src = REPO / "data/spec_datasets" / f"{DATASET}_shard{SHARD_ID}"
    out = WORK / src.name
    shutil.rmtree(out, ignore_errors=True)
    shutil.copytree(src, out)
    sh(f"du -sh {out}/*")
    print("Xong shard. Save Version -> Save & Run All để lưu output; khi đủ shard thì chạy notebook STAGE = 'merge'.")
else:
    labels = pd.read_csv(DS_DIR / "labels.tsv", sep="\t")
    specs = set(labels["spec"].astype(str))
    with h5py.File(DS_DIR / "magma_outputs/magma_tree.hdf5", "r") as f:
        tree = {k for k in f.keys() if not k.startswith("__")}
    with h5py.File(DS_DIR / "subformulae/no_subform.hdf5", "r") as f:
        sub = {k.rsplit("_collision", 1)[0] for k in f.keys()}
    print(f"labels.tsv: {len(specs)} mục | cây MAGMa: {len(tree & specs)} ({100 * len(tree & specs) / len(specs):.1f}%) "
          f"| no_subform: {len(sub & specs)} ({100 * len(sub & specs) / len(specs):.1f}%)")
    if tree - specs or sub - specs:
        raise RuntimeError("magma_outputs / subformulae có mục không thuộc labels.tsv")
    for split in sorted((DS_DIR / "splits").glob("*.tsv")):
        df = pd.read_csv(split, sep="\t")
        print(f"{split.name}: {df.iloc[:, 1].value_counts().to_dict()}")

    out = WORK / "ms-pred" / DS_REL       # cùng đường dẫn notebook train mong đợi (RESUME_FROM)
    shutil.rmtree(out, ignore_errors=True)
    out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DS_DIR, out, symlinks=False)
    sh(f"du -sh {out}/*")
    print(f"\nXong: {out}\nSave Version -> Save & Run All. Ở notebook train (marason_public_kaggle.ipynb): Add Input output này, "
          f"đặt DATASET = '{DATASET}' và RESUME_FROM = '/kaggle/input/<tên notebook này>/ms-pred'.")